# VSL400 – trích landmark BÀN TAY (hand IMAGE) toàn bộ front_view

Trước khi chạy: **Settings → Internet: On**, **Accelerator: None (CPU)**, **Add Input**: dataset `nguyenanfms/vsl-vietnamese-sign-language-v2`.
Chạy nền: **Save Version → Save & Run All (Commit)**. Output nằm ở tab *Output* của version: thư mục `hands/`.

Chạy tiếp khi hết giờ: Add Input = *output của version trước* (Notebook Output), rồi Save & Run All lại; cell 4 tự bỏ qua video đã xong.

In [ ]:
# 1. Lấy code (branch v2-model) + kiểm tra phiên bản Python hệ thống
import sys, platform, os
# Kernel đặt MPLBACKEND=matplotlib_inline; venv riêng không có -> mediapipe lỗi khi import. Mọi lệnh ! sau đây dùng Agg.
os.environ["MPLBACKEND"] = "Agg"
print("Python hệ thống:", platform.python_version(), "| CPU:", os.cpu_count())
!rm -rf /kaggle/working/repo && git clone -q -b v2-model https://github.com/tuna195/edgeai_vsign_project /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

In [ ]:
# 2. Môi trường Python 3.12 riêng (mediapipe 0.10.18 chỉ có cho Python 3.9–3.12, cần numpy 1.26.4)
#    Luôn dùng venv riêng ở /tmp (không lẫn với thư viện của Kaggle, không bị lưu vào output)
!pip install -q uv
!uv venv -q -p 3.12 /tmp/venv
!uv pip install -q --python /tmp/venv/bin/python mediapipe==0.10.18 numpy==1.26.4 opencv-contrib-python==4.11.0.86 pandas==2.2.3
!/tmp/venv/bin/python -c "import sys, mediapipe, cv2, numpy; print('venv:', sys.version.split()[0], 'mediapipe', mediapipe.__version__, 'cv2', cv2.__version__, 'numpy', numpy.__version__)"


In [ ]:
# 3. Tải 2 file .task theo URL trong extractor_config.json (load_config sẽ kiểm sha256)
import json, urllib.request
cfg = json.load(open("configs/extractor_config.json", encoding="utf-8"))
os.makedirs("/tmp/models", exist_ok=True)
for k in ["hand_model", "pose_model"]:
    dst = f"/tmp/models/{cfg[k]['file']}"
    if not os.path.exists(dst):
        urllib.request.urlretrieve(cfg[k]["url"], dst)
    print(dst, os.path.getsize(dst), "bytes")

In [ ]:
# 4. Cấu hình đường dẫn + tự tìm output cũ để chạy tiếp
import glob
ROOT = "/kaggle/input/datasets/nguyenanfms/vsl-vietnamese-sign-language-v2/raw/raw/VSL400"
if not os.path.isdir(ROOT):  # đường mount kiểu cũ
    ROOT = (glob.glob("/kaggle/input/*/raw/raw/VSL400") + glob.glob("/kaggle/input/*/*/*/raw/raw/VSL400"))[0]
OUT = "/kaggle/working/hands"
# Output của version trước (nếu đã Add Input): tìm manifest.csv ở độ sâu 1..5, không quét cả dataset
RESUME = sorted({os.path.dirname(m) for d in range(1, 6)
                 for m in glob.glob("/kaggle/input/" + "*/" * d + "manifest.csv")})
print("ROOT  :", ROOT, os.path.isdir(ROOT))
print("OUT   :", OUT)
print("RESUME:", RESUME or "(không có, chạy từ đầu)")
RESUME_ARG = ("--resume-from " + " ".join(RESUME)) if RESUME else ""

In [ ]:
# 5. Chạy thử 20 video (ghi ra /tmp, không lẫn output) -> xem tốc độ, ước tính tổng thời gian
!rm -rf /tmp/hands_test
!/tmp/venv/bin/python -m src.data.extract_hands_kaggle --root {ROOT} --model-dir /tmp/models --out-dir /tmp/hands_test --limit 20 --log-every-sec 30 2>&1 | grep -v "^W0000\|^I0000\|absl::\|inference_feedback\|XNNPACK"
import pandas as pd
m = pd.read_csv("/tmp/hands_test/manifest.csv", dtype={"video_id": str})
print(m[["video_id", "status", "n_frames", "ratio_ge1_hand", "ratio_2_hands", "hand_px_p50", "seconds"]].to_string(index=False))

In [ ]:
# 6. Chạy TOÀN BỘ (dừng sạch trước 11 giờ). Tiến độ in mỗi 10 phút.
!/tmp/venv/bin/python -m src.data.extract_hands_kaggle --root {ROOT} --model-dir /tmp/models --out-dir {OUT} {RESUME_ARG} --time-budget-hours 11 --log-every-sec 600 2>&1 | grep -v "^W0000\|^I0000\|absl::\|inference_feedback\|XNNPACK"

In [ ]:
# 7. Tóm tắt (để dán lại cho phiên M)
import pandas as pd, glob
ms = [pd.read_csv(f, dtype={"video_id": str}) for f in [f"{OUT}/manifest.csv", *[f"{d}/manifest.csv" for d in RESUME]] if os.path.exists(f)]
m = pd.concat(ms).drop_duplicates("video_id", keep="last")
ok = m[m.status == "ok"]
print("Python hệ thống:", platform.python_version())
print("Video ok:", len(ok), "| lỗi:", (m.status != "ok").sum(), "| trong json ok:", ok.in_json.sum(), "/ 24753")
print(m[m.status != "ok"][["video_id", "path", "error"]].to_string(index=False))
print("\nThời gian mỗi video (giây):"); print(ok.seconds.describe().round(2).to_string())
w = ok.n_frames
print("\nTỉ lệ frame thấy >=1 tay:", round((ok.ratio_ge1_hand * w).sum() / w.sum(), 3),
      "| thấy 2 tay:", round((ok.ratio_2_hands * w).sum() / w.sum(), 3),
      "| frame 2 tay cùng nhãn:", int(ok.n_same_label.sum()))
print("\nCỡ bàn tay p50 (px, ảnh 640x480 sau letterbox):"); print(ok.hand_px_p50.describe().round(1).to_string())
print("\nTheo Part:"); print(ok.groupby("part")[["ratio_ge1_hand", "ratio_2_hands", "hand_px_p50", "seconds"]].mean().round(3).to_string())
!ls -la {OUT} | head; du -sh {OUT}